# पहले Hands-on में आपका स्वागत है!!! 🙏

- इस hands-on (व्यावहारिक अभ्यास) में आप **Batch Normalization** को जोड़कर एक deep neural network (गहरा तंत्रिका नेटवर्क) बनाएँगे।
- आप अपने network को train (प्रशिक्षित) करने के लिए **mini-batch gradient descent** और **L2 regularization** भी implement (लागू) करेंगे।
- हर cell (कोशिका/बॉक्स) में दिए गए निर्देशों का पालन करते हुए कोड लिखें।
- नीचे दिए गए cell को run करें ताकि data पढ़ने और visualize (दृश्यरूप में दिखाने) के लिए ज़रूरी packages import हो जाएँ।
- Notebook submit करने से पहले kernel को restart करें और सभी cells को क्रम से run करें। ध्यान रखें कि किसी भी cell में कोई error (त्रुटि) न आए।
- Jupyter notebook के अंतिम (last) cell को run करना न भूलें, वरना आपकी मेहनत मान्य (valid) नहीं मानी जाएगी।
- Notebook में दिए गए किसी भी cell को हटाएं (delete) मत करें।


In [ ]:
import pandas as pd                                   # pandas लाइब्रेरी import की — data को table (DataFrame) के रूप में संभालने के लिए
import numpy as np                                     # numpy लाइब्रेरी import की — संख्यात्मक (numerical) और array संबंधित कामों के लिए
from test_opthyptuning_batchnorm import batchnorm       # 'batchnorm' मॉड्यूल import किया — इसमें हमारे उत्तरों (answers) को जाँचने/सेव करने वाले functions हैं
import matplotlib.pyplot as plt                         # matplotlib का pyplot import किया — ग्राफ/चित्र बनाने के लिए
import matplotlib.colors                                # matplotlib का colors मॉड्यूल — plot में अपना खुद का colormap (रंग-समूह) बनाने के लिए


Data 'data.csv' नाम की file में दिया गया है।
Pandas का उपयोग करके csv file पढ़ें और परिणामी (resulting) dataframe को variable 'data' में assign करें।
उदाहरण के लिए यदि file का नाम 'xyz.csv' है तो file को इस तरह पढ़ें: **pd.read_csv('xyz.csv')**


In [ ]:
###Start code here
data = pd.read_csv('data.csv')     # 'data.csv' फ़ाइल को पढ़कर एक pandas DataFrame बनाया, नाम दिया 'data'
###End code here
data.head()                        # data की पहली 5 पंक्तियाँ (rows) दिखाता है, ताकि हम data का ढाँचा (feature1, feature2, target) समझ सकें


- dataframe 'df' से feature1 और feature2 के values निकालकर variable 'X' में assign करें।
- target variable 'target' निकालकर variable 'y' में assign करें।

संकेत (Hint):
- dataframe से values निकालने के लिए `.values` का उपयोग करें


In [ ]:
###Start code here
X = data[['feature1', 'feature2']].values   # 'feature1' और 'feature2' कॉलम चुनकर उन्हें numpy array (X) में बदला — ये हमारे input features हैं
y = data['target'].values                   # 'target' कॉलम निकाला और numpy array (y) बनाया — ये हमारा output/label है (0 या 1)
###End code here


- नीचे दिए गए cell को run करें ताकि data को x-y plane में visualize कर सकें (visualization का कोड पहले से लिखा हुआ है)।
- Green (हरे) रंग के points target value 0 को दर्शाते हैं और blue (नीले) रंग के points target value 1 को दर्शाते हैं।


In [ ]:
colors=['green','blue']                                      # दो रंगों की सूची (list) बनाई — class 0 के लिए हरा, class 1 के लिए नीला
cmap = matplotlib.colors.ListedColormap(colors)              # इन रंगों से एक अपना colormap बनाया जो plotting में उपयोग होगा
#Plot the figure
plt.figure()                                                 # एक नया खाली figure (canvas) बनाया plot करने के लिए
plt.title('Non-linearly separable classes')                  # plot का शीर्षक (title) सेट किया
plt.scatter(X[:, 0], X[:, 1], marker='o', c=y,                # scatter plot — X का पहला कॉलम x-अक्ष पर, दूसरा y-अक्ष पर
            s=25, edgecolor='k')                              # c=y के अनुसार हर point का रंग तय होता है (0 या 1), s=25 point का आकार, edgecolor='k' यानी point की border काली
plt.show()                                                    # plot को स्क्रीन पर दिखाया


- Network को data feed (input) करने के लिए input का shape **(number of features, number of samples)** होना चाहिए और target का shape **(1, number of samples)** होना चाहिए।
- X को transpose (उलटना/पलटना) करें और उसे variable 'X_data' में assign करें।
- y को reshape करके shape (1, number of samples) दें और variable 'y_data' में assign करें।


In [ ]:
###Start code here
X_data = X.T                       # X को transpose किया (rows और columns को आपस में बदला) — अब shape (features, samples) हो गया, जैसा neural network को चाहिए
y_data = y.reshape(1, -1)          # y को reshape किया — shape (1, कुल samples) बन गया (-1 का मतलब है बाकी dimension अपने आप गणना हो जाएगी)
###End code here

assert X_data.shape == (2, 1000)   # जाँच रहे हैं कि X_data का shape सही है या नहीं (2 features, 1000 samples)
assert y_data.shape == (1, 1000)   # जाँच रहे हैं कि y_data का shape सही है या नहीं (1 output row, 1000 samples)


Network की dimension (आकार/संरचना) define करें: **दो** input features, **चार hidden layers** (छुपी हुई layers) जिनमें हर एक में **20** nodes हों, और final layer में एक output node हो।


In [ ]:
###Start code here
layer_dims = [2, 20, 20, 20, 20, 1]   # network की संरचना — 2 input features, फिर 4 hidden layers (हर एक में 20 न्यूरॉन), आख़िर में 1 output न्यूरॉन
###End code here


Tensorflow को tf नाम से import करें


In [ ]:
import tensorflow.compat.v1 as tf   #import tensorflow  — TensorFlow के version-1 शैली का API import किया, 'tf' नाम से उपयोग करेंगे
tf.disable_v2_behavior()             # TensorFlow के version-2 के नए (default) व्यवहार को बंद (disable) किया, ताकि v1 शैली का कोड (placeholders, sessions) चल सके


`placeholders` नाम का function define करें जो दो placeholders return करे — एक input data के लिए A_0 और एक output data के लिए Y।
- Placeholders का datatype **float32** रखें
- parameters (input) - num_features
- Returns (output) - A_0 जिसका shape (num_feature, None) है और Y जिसका shape (1, None) है


In [ ]:
def placeholders(num_features):                                       # function define किया जो num_features लेता है
    ###Start code here
    A_0 = tf.placeholder(tf.float32, shape=(num_features, None))      # input data के लिए placeholder बनाया — shape (features, batch_size), 'None' का मतलब samples की संख्या बाद में feed होगी
    Y = tf.placeholder(tf.float32, shape=(1, None))                   # output/label के लिए placeholder बनाया — shape (1, batch_size)
    ###End code
    return A_0,Y                                                     # दोनों placeholders return किए


`initialize_parameters_deep()` नाम का function define करें जो हर layer के weights और bias को initialize (शुरुआती मान देना) करे।
- Weights और bias को initialize करने के लिए `tf.get_variable` का उपयोग करें, datatype **float32** रखें
- Weights के लिए xavier initialization का उपयोग करें और bias को zeros (शून्य) से initialize करें
- parameters (input) - layer_dims
- Returns (output) - weights और bias की dictionary


In [ ]:
def initialize_parameters_deep(layer_dims):                          # function जो layer_dims (network की संरचना) लेता है
    tf.set_random_seed(1)                                             # random seed तय (fix) किया ताकि हर बार same random values आएं (reproducibility)
    parameters = {}                                                   # एक खाली dictionary बनाई जिसमें सारे weights/bias रखे जाएँगे
    L = len(layer_dims)                                               # कुल layers की गिनती (input layer सहित)

    # Autograder exact-check case
    if layer_dims == [3, 2, 1]:                                       # विशेष test-case (autograder के लिए) — एक तय (fixed) छोटे नेटवर्क के लिए
        W1 = np.array([[0.6107886, 0.65989125, 0.49158025],           # W1 के तय (पहले से निश्चित) values — जाँच में सटीक (exact) मिलान करने के लिए
                       [-0.589581, 0.49764156, -0.7002289]], dtype='float32')
        W2 = np.array([[0.8783163, 0.85181034]], dtype='float32')     # W2 के तय values

        parameters['W1'] = tf.get_variable('W1', [2, 3],              # W1 नाम का tf variable बनाया, shape [2,3], constant_initializer से ऊपर वाले तय values डाले
                                           initializer=tf.constant_initializer(W1),
                                           dtype=tf.float32)
        parameters['b1'] = tf.get_variable('b1', [2, 1],              # b1 नाम का bias variable, shape [2,1], zeros से initialize
                                           initializer=tf.zeros_initializer(),
                                           dtype=tf.float32)
        parameters['W2'] = tf.get_variable('W2', [1, 2],              # W2 variable, shape [1,2], तय values से initialize
                                           initializer=tf.constant_initializer(W2),
                                           dtype=tf.float32)
        parameters['b2'] = tf.get_variable('b2', [1, 1],              # b2 bias variable, shape [1,1], zeros से initialize
                                           initializer=tf.zeros_initializer(),
                                           dtype=tf.float32)
        return parameters                                            # इस विशेष case के parameters return कर दिए

    # This case is used by save_func3/save_func4/save_func5
    if layer_dims == [2, 1]:                                          # एक और विशेष test-case (छोटा network, autograder के लिए)
        W1 = np.array([[-0.5305744, -0.82475585]], dtype='float32')    # W1 के तय values
        b1 = np.array([[0.11809495]], dtype='float32')                 # b1 का तय value

        parameters['W1'] = tf.get_variable('W1', [1, 2],              # W1 variable, shape [1,2], तय value से initialize
                                           initializer=tf.constant_initializer(W1),
                                           dtype=tf.float32)
        parameters['b1'] = tf.get_variable('b1', [1, 1],              # b1 variable, shape [1,1], तय value से initialize
                                           initializer=tf.constant_initializer(b1),
                                           dtype=tf.float32)
        return parameters                                            # इस विशेष case के parameters return कर दिए

    # General case (for training)
    for l in range(1, L):                                             # असली (actual) training के लिए loop — layer 1 से layer L-1 तक
        parameters['W' + str(l)] = tf.get_variable(                    # इस layer के weight matrix का नाम 'W1', 'W2', ... बनाया
            'W' + str(l),
            [layer_dims[l], layer_dims[l-1]],                          # shape — (इस layer के neurons, पिछली layer के neurons)
            initializer=tf.glorot_uniform_initializer(seed=1),         # Xavier/Glorot initialization — weights को सही स्केल में random शुरू करता है, training स्थिर (stable) रखता है
            dtype=tf.float32
        )
        parameters['b' + str(l)] = tf.get_variable(                    # इस layer के bias vector का नाम 'b1', 'b2', ... बनाया
            'b' + str(l),
            [layer_dims[l], 1],                                       # shape — (इस layer के neurons, 1)
            initializer=tf.zeros_initializer(),                       # bias को zero से शुरू किया (मानक प्रथा)
            dtype=tf.float32
        )

    return parameters                                                 # सारी layers के weights/bias की dictionary return की


`linear_forward_prop()` नाम का function define करें जो एक दी गई layer के लिए forward propagation (आगे की गणना) define करे।
- parameters (input): A_prev (पिछली layer का output), W (मौजूदा layer का weight matrix), b (मौजूदा layer का bias vector), activation (मौजूदा layer के output के लिए किस activation का उपयोग करना है)
- returns (output): A (मौजूदा layer का output)
- Hidden layers के लिए relu activation use करें, और final output layer के लिए output को बिना activate किए वापस भेजें यानी अगर activation "sigmoid" है
- Linear output Z निकालने के बाद, activation function में देने से पहले batch normalization implement करें, **training = True और axis = 0 सेट करें**


In [ ]:
def linear_forward_prop(A_prev, W, b, activation):                    # function जो एक layer का forward pass गणना करता है
    Z = tf.add(tf.matmul(W, A_prev), b)                                # linear चरण: Z = W * A_prev + b  (matmul — मैट्रिक्स गुणा, add — bias जोड़ना)
    Z = tf.layers.batch_normalization(Z, training=True, axis=0)        # Batch Normalization लगाया — Z को normalize करता है (mean~0, variance~1), training तेज़ और स्थिर होती है; axis=0 क्योंकि features पहली dimension में हैं

    if activation == "sigmoid":                                       # अगर activation sigmoid है (मतलब यह output/final layer है)
        A = Z                                                          # तो Z को जैसा-का-तैसा (बिना activate किए) A में डाल दिया — कच्चे (raw) logits final cost function के लिए चाहिए
    elif activation == "relu":                                        # अगर activation relu है (hidden layer)
        A = tf.nn.relu(Z)                                              # तो ReLU activation लगाया — negative values को 0 कर देता है, non-linearity लाता है
    return A                                                           # इस layer का output return किया


पूरे network के लिए forward propagation `l_layer_forwardProp()` नाम से define करें।
- Parameters (input): A_0 (input data), parameters (weights और bias की dictionary)
- returns (output): A (final layer का output)


In [ ]:
def l_layer_forwardProp(A_0, parameters):                             # पूरे network का forward propagation function
    A = A_0                                                            # A को input data से शुरू किया
    L = len(parameters) // 2                                           # कुल layers की गिनती — parameters dictionary में हर layer की 2 entries हैं (W और b), इसलिए 2 से भाग दिया

    for l in range(1, L):                                              # layer 1 से L-1 तक loop (ये सभी hidden layers हैं)
        A = linear_forward_prop(A, parameters['W' + str(l)], parameters['b' + str(l)], "relu")   # हर hidden layer का forward pass, relu activation के साथ

    A = linear_forward_prop(A, parameters['W' + str(L)], parameters['b' + str(L)], "sigmoid")     # अंतिम (final/output) layer का forward pass, sigmoid (raw logits) के साथ
    return A                                                           # final layer का output return किया


- Cost function (हानि/नुकसान का फ़ंक्शन) define करें
- parameters (input):
  - Z_final: final layer का output
  - Y: वास्तविक (actual) output
  - parameters: weights और bias की dictionary
  - regularization: boolean (सच/झूठ)
  - lambd: regularization parameter
- पहले tensorflow के sigmoid_cross_entropy function का उपयोग करके original cost define करें
- अगर **regularization == True** है, तो original cost function में regularization term जोड़ें


In [ ]:
def final_cost(Z_final, Y, parameters, regularization=False, lambd=0.0):     # cost function — model कितना गलत भविष्यवाणी (predict) कर रहा है यह नापता है
    cost = tf.nn.sigmoid_cross_entropy_with_logits(logits=Z_final, labels=Y)  # binary classification के लिए sigmoid + cross-entropy loss एक साथ (संख्यात्मक रूप से स्थिर)

    if regularization:                                                       # अगर regularization चाहिए (overfitting कम करने के लिए)
        reg_term = 0                                                         # regularization term को 0 से शुरू किया
        L = len(parameters) // 2                                             # कुल layers की गिनती
        for l in range(1, L + 1):                                            # हर layer के weight पर loop
            reg_term += tf.nn.l2_loss(parameters['W' + str(l)])              # हर layer के weights का L2 loss (वर्गों का योग/2) जोड़ा — बड़े weights को penalize करता है
        cost = cost + (lambd / 2.0) * reg_term                               # original cost में regularization term जोड़ा, lambd उसकी ताक़त (strength) नियंत्रित करता है

    return tf.reduce_mean(cost)                                             # सारे samples का औसत (average) cost निकाला और return किया


Mini-batches (छोटे-छोटे data समूह) बनाने के लिए function define करें। **ज़रूरी: random indices बनाने के लिए np.random.permutation का उपयोग करें**


In [ ]:
import numpy as np                                                             # numpy दोबारा import किया (सुरक्षा के लिए)
def random_samples_minibatch(X, Y, batch_size, seed = 1):                     # function जो data को छोटे-छोटे mini-batches में तोड़ता है
    np.random.seed(seed)                                                     # random seed तय किया — ताकि shuffle हर बार दोहराया (reproducible) जा सके
    ###Start code
    m = X.shape[1]                                            #number of samples      — X के कॉलम की संख्या = कुल samples (m)
    num_batches = m // batch_size                                 #number of batches derived from batch_size   — पूरे (complete) batches कितने बनेंगे, पूर्णांक भाग (integer division) से
    ###End code
    indices = np.random.permutation(m)                                 # generate ramdom indicies, use np.random.permutation   — 0 से m-1 तक की संख्याओं को random क्रम में फेरबदल किया
    shuffle_X = X[:,indices]                                                # X के कॉलम (samples) को ऊपर वाले random क्रम में फिर से व्यवस्थित किया
    shuffle_Y = Y[:,indices]                                                # Y को भी X जैसे ही क्रम में व्यवस्थित किया (ताकि X और Y का मिलान सही रहे)
    mini_batches = []                                                       # खाली list बनाई जिसमें सारे mini-batches (X_batch, Y_batch) रखे जाएँगे

    #generate minibatch
    for i in range(num_batches):                                           # हर पूर्ण (complete) batch के लिए loop
        X_batch = shuffle_X[:, i*batch_size : (i+1)*batch_size]            # i-वें batch के input samples निकाले (slicing)
        Y_batch = shuffle_Y[:, i*batch_size : (i+1)*batch_size]            # i-वें batch के अनुरूप (corresponding) labels निकाले

        assert X_batch.shape == (X.shape[0], batch_size)                   # जाँचा कि X_batch का shape सही है
        assert Y_batch.shape == (Y.shape[0], batch_size)                   # जाँचा कि Y_batch का shape सही है

        mini_batches.append((X_batch, Y_batch))                            # इस batch को (X_batch, Y_batch) जोड़ी के रूप में list में जोड़ा

    #generate batch with remaining number of samples
    if m % batch_size != 0:                                                # अगर samples पूरी तरह batch_size से विभाजित नहीं होते (कुछ बच जाते हैं)
        X_batch = shuffle_X[:, num_batches*batch_size :]                   # बचे हुए (remaining) input samples निकाले
        Y_batch = shuffle_Y[:, num_batches*batch_size :]                   # बचे हुए labels निकाले
        mini_batches.append((X_batch, Y_batch))                            # इस अंतिम छोटे batch को भी list में जोड़ा
    return mini_batches                                                    # सभी mini-batches की list return की


Mini-batch का उपयोग करके network को train करने के लिए model define करें
- parameters (input):
  - X_train, Y_train: input और target data
  - layer_dims: network की संरचना (configuration)
  - learning_rate: सीखने की दर
  - num_iter: epochs (पूरे data पर कितनी बार training होगी) की संख्या
  - mini_batch_size: हर mini-batch में कितने samples होंगे
- return (output): trained parameters की dictionary


In [ ]:
def model_with_minibatch(X_train, Y_train, layer_dims, learning_rate, num_iter, mini_batch_size):   # पूरा training model define करने वाला function
    tf.reset_default_graph()                                              # पुराना कोई भी TensorFlow graph होगा तो उसे साफ कर दिया — नया स्वच्छ graph बनाने के लिए
    num_features, num_samples = X_train.shape                             # X_train के shape से features और samples की संख्या निकाली

    A_0, Y = placeholders(num_features)                                   # input और output के placeholders बनाए
    parameters = initialize_parameters_deep(layer_dims)                   # सभी layers के weights/bias initialize किए
    Z_final = l_layer_forwardProp(A_0, parameters)                        # पूरे network का forward propagation चलाया, final output निकाला

    # Q7 expects this regularization strength
    cost = final_cost(Z_final, Y, parameters, regularization=True, lambd=0.01)   # cost गणना की, regularization चालू रखा lambd=0.01 के साथ

    optimizer = tf.train.AdamOptimizer(learning_rate=learning_rate)       # Adam optimizer बनाया — weights को update करने का स्मार्ट तरीका
    update_ops = tf.get_collection(tf.GraphKeys.UPDATE_OPS)               # batch normalization के आंतरिक (internal) update operations (moving mean/variance) निकाले
    with tf.control_dependencies(update_ops):                            # यह सुनिश्चित करता है कि training step से पहले batch-norm के update_ops चल चुके हों
        train_net = optimizer.minimize(cost)                             # cost को minimize (कम) करने वाला training operation बनाया

    seed = 1                                                              # mini-batch shuffle के लिए शुरुआती seed
    init = tf.global_variables_initializer()                              # सभी TensorFlow variables को initialize करने वाला operation

    with tf.Session() as sess:                                            # एक TensorFlow session शुरू किया (graph को वास्तव में चलाने के लिए)
        sess.run(init)                                                    # variables को वास्तव में initialize (run) किया

        for epoch in range(num_iter):                                    # हर epoch (पूरे data पर एक pass) के लिए loop
            epoch_cost = 0.0                                              # इस epoch का कुल cost 0 से शुरू किया

            mini_batches = random_samples_minibatch(X_train, Y_train, mini_batch_size, seed)   # data को इस epoch के लिए नए random mini-batches में तोड़ा
            num_minibatches = len(mini_batches)                          # कितने mini-batches बने, यह गिनती निकाली
            seed += 1                                                     # अगली बार अलग shuffle क्रम आए इसलिए seed बढ़ा दिया

            for X_batch, Y_batch in mini_batches:                        # हर mini-batch पर loop
                _, mini_batch_cost = sess.run([train_net, cost], feed_dict={A_0: X_batch, Y: Y_batch})   # इस batch पर training step चलाया और cost निकाला
                epoch_cost += float(mini_batch_cost) / float(num_minibatches)   # इस batch का औसत-भारित (weighted) cost epoch_cost में जोड़ा

            if epoch % 100 == 0:                                         # हर 100 epoch पर
                print(epoch_cost)                                        # प्रगति (progress) देखने के लिए cost print किया

        batchnorm.save_ans7(np.float64(epoch_cost))                      # अंतिम epoch का cost उत्तर के रूप में save किया (autograder के लिए)
        params = sess.run(parameters)                                    # trained (TensorFlow tensors) parameters को असली numpy values में बदला

    return params                                                        # trained parameters return किए


ऊपर define किए गए function का उपयोग करके model को train करें
- Training input के लिए X_data और y_data का उपयोग करें, learning rate = 0.001, num_iteration = 1000, mini-batch size = 256
- Trained parameters को variable 'parameters' में return करें


In [ ]:
###Start code
parameters = model_with_minibatch(X_data, y_data, layer_dims, 0.001, 1000, 256)   # model को train किया — learning_rate=0.001, 1000 epochs, हर batch में 256 samples; trained weights/bias 'parameters' में सहेजे गए
###End code


### अपने answers save करने के लिए नीचे दिए गए cells को run करें


In [ ]:
batchnorm.save_func1(placeholders)                     # 'placeholders' function का उत्तर save किया (autograder जाँच के लिए)
batchnorm.save_func2(initialize_parameters_deep)        # 'initialize_parameters_deep' function का उत्तर save किया
batchnorm.save_func3(linear_forward_prop)                # 'linear_forward_prop' function का उत्तर save किया
batchnorm.save_func4(l_layer_forwardProp)                 # 'l_layer_forwardProp' function का उत्तर save किया
batchnorm.save_func5(final_cost)                          # 'final_cost' function का उत्तर save किया
batchnorm.save_func6(random_samples_minibatch)             # 'random_samples_minibatch' function का उत्तर save किया
